# PHI De-Identification Accelerator — One-Click Launcher

> ⚠️ **SYNTHETIC DATA ONLY.** This launcher stands up the accelerator on **synthetic**
> Epic-Caboodle data. It is a reference / blueprint pattern, **not** a certified
> de-identification service. Before any real PHI, work through
> [`docs/pre_real_phi_checklist.md`](docs/pre_real_phi_checklist.md).

**What this does (Run All, ~5–10 min):** automates every manual step in the
[QUICKSTART](QUICKSTART.md) so you don't have to click through the portal:

1. Uses **the workspace you imported this notebook into as the Raw workspace** (no throwaway
   workspace) and creates the other **two** (Analytics / Vault) on the same capacity.
2. Creates a **Lakehouse** in each (`lh_raw`, `lh_analytics`, `lh_vault`).
3. Downloads this repo from GitHub and uploads `src/` + `config/` to
   `Files/accelerator/` in each Lakehouse.
4. Uploads the 13 synthetic Caboodle CSVs to the Raw Lakehouse at
   `Files/raw/caboodle_provider/`.
5. **Imports the notebooks** into the correct workspaces, auto-patching the
   cross-workspace config (workspace + lakehouse names) so there are **no manual GUID edits**.

**After it finishes** you set the tokenization pepper and run the notebooks in order
(the final cell prints the exact sequence). The manual QUICKSTART path still works if you
prefer to click through it yourself — this launcher is an **option**, not a replacement.

### How to run this
1. **Create a new workspace** and name it **`PHI-Raw`** — this becomes your Raw / PHI workspace.
   Make sure it's on an **F2+ / Trial** capacity.
2. **Import this notebook into that `PHI-Raw` workspace** (Workspace → New item → Import notebook).
3. Click **Run All**. The launcher keeps `PHI-Raw` as Raw and creates the other two workspaces
   (`PHI-Analytics`, `PHI-Vault`) for you — no extra/throwaway workspace.

### Prerequisites
- You must be able to **create workspaces** (Fabric admin setting: *Users can create workspaces*).
- Internet access to `github.com` (or fork to an internal location and set `GITHUB_OWNER`).

## 1. Configuration
Edit only if you want non-default names or a specific capacity. Defaults just work.

In [ ]:
# ---- GitHub source (public repo — no token needed) ----
GITHUB_OWNER  = "rasgiza"
GITHUB_REPO   = "fabric-phi-deidentification-accelerator"
GITHUB_BRANCH = "main"
GITHUB_TOKEN  = ""          # only needed if you fork to a PRIVATE repo

# ---- Workspace + lakehouse names (created if missing, reused if present) ----
WS_RAW        = "PHI-Raw"
WS_ANALYTICS  = "PHI-Analytics"
WS_VAULT      = "PHI-Vault"
LH_RAW        = "lh_raw"
LH_ANALYTICS  = "lh_analytics"
LH_VAULT      = "lh_vault"

# ---- Capacity: leave blank to auto-pick the first ACTIVE capacity you can see ----
CAPACITY_ID   = ""

# ---- Behaviour flags ----
USE_CURRENT_WORKSPACE = True    # True = use THIS notebook's workspace as Raw (no extra workspace); False = create WS_RAW too
SINGLE_WORKSPACE = False   # True = put all 3 lakehouses in ONE workspace (quick demo; weakens the isolation story)
UPLOAD_DATA      = True    # upload the 13 synthetic Caboodle CSVs to lh_raw
IMPORT_NOTEBOOKS = True    # import + auto-patch the notebooks into their workspaces
INCLUDE_BEFORE   = True    # also import 03_gold_star (the 'before' PHI-in-Gold notebook for Demo Act 1)

print("Config loaded. Raw =", WS_RAW, "| Analytics =", WS_ANALYTICS, "| Vault =", WS_VAULT,
      "| single-workspace =", SINGLE_WORKSPACE)

## 2. Helpers (Fabric REST + OneLake)

In [ ]:
import base64, io, json, os, re, time, zipfile, urllib.request
import requests
import notebookutils

FABRIC_API = "https://api.fabric.microsoft.com/v1"
_TOKEN = notebookutils.credentials.getToken("pbi")
H = {"Authorization": f"Bearer {_TOKEN}", "Content-Type": "application/json"}


def _lro(resp, what="operation"):
    """Return the created/updated resource, following an async (202) Location if needed."""
    if resp.status_code in (200, 201):
        return resp.json() if resp.text else {}
    if resp.status_code == 202:
        op = resp.headers.get("Location")
        for _ in range(60):
            time.sleep(3)
            st = requests.get(op, headers=H)
            state = st.json().get("status") if st.text else None
            if state == "Succeeded":
                res = requests.get(op + "/result", headers=H)
                return res.json() if res.text else {}
            if state == "Failed":
                raise RuntimeError(f"{what} failed: {st.text}")
        raise TimeoutError(f"{what} did not complete in time")
    raise RuntimeError(f"{what} HTTP {resp.status_code}: {resp.text}")


def pick_capacity():
    if CAPACITY_ID:
        return CAPACITY_ID
    caps = requests.get(f"{FABRIC_API}/capacities", headers=H).json().get("value", [])
    active = [c for c in caps if c.get("state", "").lower() == "active"]
    if not active:
        raise RuntimeError("No ACTIVE capacity found. Set CAPACITY_ID in the config cell.")
    print(f"Using capacity: {active[0]['displayName']} ({active[0]['id']})")
    return active[0]["id"]


def get_or_create_workspace(name, capacity_id):
    existing = requests.get(f"{FABRIC_API}/workspaces", headers=H).json().get("value", [])
    for w in existing:
        if w["displayName"] == name:
            print(f"  • workspace exists: {name} ({w['id']})")
            requests.post(f"{FABRIC_API}/workspaces/{w['id']}/assignToCapacity",
                          headers=H, json={"capacityId": capacity_id})
            return w["id"]
    body = {"displayName": name, "capacityId": capacity_id}
    w = _lro(requests.post(f"{FABRIC_API}/workspaces", headers=H, json=body), f"create workspace {name}")
    print(f"  • workspace created: {name} ({w['id']})")
    return w["id"]


def get_or_create_lakehouse(ws_id, name):
    existing = requests.get(f"{FABRIC_API}/workspaces/{ws_id}/lakehouses", headers=H).json().get("value", [])
    for lh in existing:
        if lh["displayName"] == name:
            print(f"  • lakehouse exists: {name} ({lh['id']})")
            return lh["id"]
    lh = _lro(requests.post(f"{FABRIC_API}/workspaces/{ws_id}/lakehouses", headers=H,
                            json={"displayName": name}), f"create lakehouse {name}")
    print(f"  • lakehouse created: {name} ({lh['id']})")
    return lh["id"]


def onelake_files_root(ws_id, lh_id):
    return f"abfss://{ws_id}@onelake.dfs.fabric.microsoft.com/{lh_id}.Lakehouse/Files"


def upload_dir(local_dir, abfss_dir):
    """Copy a local directory tree into OneLake Files/ (per-file, robust across schemes)."""
    n = 0
    for root, _dirs, files in os.walk(local_dir):
        for fn in files:
            if fn.endswith((".pyc",)) or "__pycache__" in root:
                continue
            lp = os.path.join(root, fn)
            rel = os.path.relpath(lp, local_dir).replace(os.sep, "/")
            notebookutils.fs.cp(f"file:{lp}", f"{abfss_dir}/{rel}", True)
            n += 1
    print(f"    uploaded {n} files -> {abfss_dir}")


print("Helpers ready.")

## 3. Download the repo from GitHub

In [ ]:
zip_url = f"https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}/archive/refs/heads/{GITHUB_BRANCH}.zip"
req = urllib.request.Request(zip_url)
if GITHUB_TOKEN:
    req.add_header("Authorization", f"token {GITHUB_TOKEN}")
print("Downloading", zip_url)
data = urllib.request.urlopen(req).read()
zf = zipfile.ZipFile(io.BytesIO(data))
extract_root = "/tmp/phi_deid_repo"
if os.path.exists(extract_root):
    import shutil; shutil.rmtree(extract_root)
zf.extractall(extract_root)
REPO = os.path.join(extract_root, f"{GITHUB_REPO}-{GITHUB_BRANCH}")
assert os.path.isdir(os.path.join(REPO, "src")), f"src/ not found under {REPO}"
print("Extracted to", REPO)

## 4. Create workspaces + lakehouses

In [ ]:
cap = pick_capacity()

print("Workspaces:")
if USE_CURRENT_WORKSPACE:
    _ctx = notebookutils.runtime.context
    ws_raw = _ctx.get("currentWorkspaceId") or _ctx.get("workspaceId")
    WS_RAW = _ctx.get("currentWorkspaceName") or WS_RAW   # use the real name for cross-workspace config patching
    print(f"  • using CURRENT workspace as Raw: {WS_RAW} ({ws_raw})")
else:
    ws_raw = get_or_create_workspace(WS_RAW, cap)
if SINGLE_WORKSPACE:
    ws_analytics = ws_vault = ws_raw
    WS_ANALYTICS = WS_VAULT = WS_RAW   # so config-patching points everything at the one workspace
    print("  (single-workspace mode: Analytics + Vault reuse the Raw workspace)")
else:
    ws_analytics = get_or_create_workspace(WS_ANALYTICS, cap)

    ws_vault = get_or_create_workspace(WS_VAULT, cap)lh_vault_id     = get_or_create_lakehouse(ws_vault, LH_VAULT)

lh_analytics_id = get_or_create_lakehouse(ws_analytics, LH_ANALYTICS)

print("Lakehouses:")lh_raw_id       = get_or_create_lakehouse(ws_raw, LH_RAW)

## 5. Upload code, config, and sample data

In [ ]:
# Upload src/ + config/ to Files/accelerator in EACH lakehouse (notebooks import from the attached default).
targets = {
    LH_RAW:       (ws_raw, lh_raw_id),
    LH_ANALYTICS: (ws_analytics, lh_analytics_id),
    LH_VAULT:     (ws_vault, lh_vault_id),
}
for name, (ws_id, lh_id) in targets.items():
    files_root = onelake_files_root(ws_id, lh_id)
    print(f"Uploading accelerator package to {name} ...")
    upload_dir(os.path.join(REPO, "src"), f"{files_root}/accelerator/src")
    upload_dir(os.path.join(REPO, "config"), f"{files_root}/accelerator/config")

# Upload the 13 synthetic Caboodle CSVs to the RAW lakehouse only.
if UPLOAD_DATA:
    data_dir = os.path.join(REPO, "sample_data", "caboodle_provider")
    raw_files_root = onelake_files_root(ws_raw, lh_raw_id)
    print("Uploading synthetic Caboodle CSVs to lh_raw ...")
    upload_dir(data_dir, f"{raw_files_root}/raw/caboodle_provider")

## 6. Import notebooks (auto-patched — no manual GUID edits)

In [ ]:
def _patch_assignment(src, var, value):
    """Rewrite a top-level `VAR = \"...\"` assignment to VAR = \"value\"."""
    pat = re.compile(rf'^(\s*{var}\s*=\s*)"[^"]*"', re.M)
    return pat.sub(lambda m: f'{m.group(1)}"{value}"', src)


def _patch_notebook_json(nb_json, patches):
    for cell in nb_json.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        src = "".join(cell.get("source", []))
        new = src
        for var, val in patches.items():
            new = _patch_assignment(new, var, val)
        if new != src:
            cell["source"] = new.splitlines(keepends=True)
    return nb_json


def import_notebook(ws_id, display_name, local_ipynb, patches=None):
    with open(local_ipynb, "r", encoding="utf-8") as fh:
        nb = json.load(fh)
    if patches:
        nb = _patch_notebook_json(nb, patches)
    payload = base64.b64encode(json.dumps(nb).encode("utf-8")).decode("ascii")
    body = {
        "displayName": display_name,
        "definition": {
            "format": "ipynb",
            "parts": [{"path": "notebook-content.ipynb", "payload": payload, "payloadType": "InlineBase64"}],
        },
    }
    _lro(requests.post(f"{FABRIC_API}/workspaces/{ws_id}/notebooks", headers=H, json=body),
         f"import notebook {display_name}")
    print(f"  • imported {display_name}")


if IMPORT_NOTEBOOKS:
    nbdir = os.path.join(REPO, "notebooks")
    # Cross-workspace config injected so the notebooks resolve each other with NO manual edits.
    patch_03b = {"SOURCE_WORKSPACE": WS_RAW, "SOURCE_LAKEHOUSE": LH_RAW}
    patch_reid = {"RAW_WORKSPACE": WS_RAW, "RAW_LAKEHOUSE": LH_RAW,
                  "ANALYTICS_WORKSPACE": WS_ANALYTICS, "ANALYTICS_LAKEHOUSE": LH_ANALYTICS}

    print("Raw workspace:")
    import_notebook(ws_raw, "01_bronze_ingest", os.path.join(nbdir, "01_bronze_ingest.ipynb"))
    import_notebook(ws_raw, "02_silver_conform", os.path.join(nbdir, "02_silver_conform.ipynb"))
    import_notebook(ws_raw, "02b_silver_deid", os.path.join(nbdir, "02b_silver_deid.ipynb"))
    if INCLUDE_BEFORE:
        import_notebook(ws_raw, "03_gold_star", os.path.join(nbdir, "03_gold_star.ipynb"))

    print("Analytics workspace:")
    import_notebook(ws_analytics, "03b_gold_safe_analytics",
                    os.path.join(nbdir, "03b_gold_safe_analytics.ipynb"), patch_03b)
    import_notebook(ws_analytics, "NB_scorecard", os.path.join(nbdir, "NB_scorecard.ipynb"))

    print("Vault workspace:")
    import_notebook(ws_vault, "NB_reidentify", os.path.join(nbdir, "NB_reidentify.ipynb"), patch_reid)

## 7. Done — next steps

In [ ]:
print("=" * 70)
print("DEPLOYMENT COMPLETE")
print("=" * 70)
print(f"Raw workspace       : {WS_RAW}       (lakehouse {LH_RAW})")
print(f"Analytics workspace : {WS_ANALYTICS} (lakehouse {LH_ANALYTICS})")
print(f"Vault workspace     : {WS_VAULT}     (lakehouse {LH_VAULT})")
print()
print("NEXT STEPS")
print("-" * 70)
print("1. Provide the tokenization pepper (synthetic demo = easiest):")
print("     In each workspace, set env var PHI_DEID_PEPPER to any non-empty string,")
print("     or configure PHI_DEID_KEYVAULT_URL for the production path.")
print("2. Attach the RIGHT default lakehouse to each notebook before running it:")
print(f"     Raw notebooks       -> {LH_RAW}")
print(f"     Analytics notebooks -> {LH_ANALYTICS}")
print(f"     Vault notebook      -> {LH_VAULT}")
print("3. Run the notebooks in order:")
print("     Raw:       01_bronze_ingest -> 02_silver_conform -> 02b_silver_deid")
print("     Analytics: 03b_gold_safe_analytics -> NB_scorecard   (expect PASS: 0/18)")
print("     Vault:     NB_reidentify  (only for a governed, approved re-identification)")
print()
print("Optional 'before' demo (Act 1): run 03_gold_star in Raw to show PHI reaching Gold.")
print("Full guided demo: docs/demo_runbook.md")